# 01 · Building the dataset suite straight from the CSO API

This notebook **reproduces the entire data pipeline** used in this project:

1. It calls the CSO PxStat API (ws.cso.ie) and downloads the raw **JSON-stat** responses;
2. It parses them into tidy tables and cleans them;
3. It derives the analysis-ready datasets (including the integrated quarterly panel).

Tables fetched (all from CSO, Housing / Prices themes):

| Code  | Table | Frequency | Role in project |
|-------|-------|-----------|-----------------|
| HPM09 | Residential Property Price Index | Monthly 2005-2026 | **Primary**: prices, national + regions |
| NDM01 | Monthly New Dwelling Completions | Monthly 2011-2026 | Supply by dwelling type |
| NDQ05 | New Dwelling Completions by area | Quarterly 2011-2026 | Supply by local authority |
| HPA09 | Residential Dwelling Property Transactions | Annual 2010-2025 | Volumes and euro prices (mean/median) |
| HSM13/HSM14 | Commencements (notices / units) | Monthly | Lead indicator (2024-25 definition break; not trended) |
| CPM24 | Consumer Price Index, All Items | Monthly | Deflator for real prices |
| CPA01 | Consumer Price Index (annual) | Annual | Context |

API pattern:  https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/CODE/JSON-stat/2.0/en

In [1]:
# %% setup + shared paths
import os, sys, json, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data'
RAW = DATA / 'raw' / 'json'
PROC = DATA / 'processed'
DER = DATA / 'derived'
for d in (RAW, PROC, DER, ROOT / 'charts'):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 60)
print('repo root:', ROOT)

repo root: /home/rob/codes/irish_housing_market


In [2]:
# %% fetch all tables from the live CSO API
BASE = 'https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/{code}/JSON-stat/2.0/en'
TABLES = ['HPM09', 'NDM01', 'NDQ05', 'HPA09', 'HSM13', 'HSM14', 'CPM24', 'CPA01']

def fetch(code):
    url = BASE.format(code=code)
    for attempt in range(4):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (research)'})
            with urllib.request.urlopen(req, timeout=240) as r:
                return r.read()
        except Exception as exc:
            print('  retry', code, str(exc)[:100])
            time.sleep(2 * (attempt + 1))
    raise RuntimeError('failed: ' + code)

sizes = {}
for code in TABLES:
    body = fetch(code)
    (RAW / (code + '.json')).write_bytes(body)
    sizes[code] = len(body)
    print('%-6s fetched: %8d bytes' % (code, len(body)))
print()
print('raw JSON-stat files now stored in data/raw/json/')

HPM09  fetched:   110994 bytes
NDM01  fetched:    12638 bytes
NDQ05  fetched:    17949 bytes
HPA09  fetched:   131096 bytes
HSM13  fetched:    77756 bytes
HSM14  fetched:    48340 bytes
CPM24  fetched:  1781261 bytes
CPA01  fetched:    24861 bytes

raw JSON-stat files now stored in data/raw/json/


In [3]:
# %% rebuild processed + derived datasets from the downloaded raw files
from build_datasets import clean_hpm09, clean_ndm01, clean_ndq05, clean_cpm24, build_panel
from build_tx import build as build_tx

rppi = clean_hpm09()
comp = clean_ndm01()
clean_ndq05()
cpi = clean_cpm24()
annual, wide = build_tx()

# derived monthly RPPI file
rppi[['date', 'period', 'series', 'region', 'property_type', 'index', 'pct_1m', 'pct_3m', 'pct_12m']].to_csv(
    DER / 'rppi_monthly_index.csv', index=False)
comp.to_csv(DER / 'new_dwelling_completions_monthly.csv', index=False)

panel = build_panel(rppi, comp, cpi)
merge_cols = ['tx_volume', 'tx_value_eur_m', 'tx_mean_price', 'tx_median_price',
              'mean_price_new', 'median_price_new', 'mean_price_existing', 'median_price_existing']
panel = panel.merge(annual[['year'] + merge_cols], on='year', how='left')
panel = panel[panel['quarter'] <= '2026Q2'].reset_index(drop=True)
panel.to_csv(DER / 'housing_market_panel_quarterly.csv', index=False)
print('quarterly panel: %d quarters, %s -> %s' % (len(panel), panel['quarter'].iloc[0], panel['quarter'].iloc[-1]))
print('derived files:')
for f in sorted(DER.glob('*.csv')):
    print('   ', f.name)

quarterly panel: 62 quarters, 2011Q1 -> 2026Q2
derived files:
    housing_market_panel_quarterly.csv
    new_dwelling_completions_monthly.csv
    rppi_monthly_index.csv
    transactions_annual_national.csv
    transactions_annual_summary.csv


In [4]:
# %% quality control checks
# (a) monthly (NDM01) vs quarterly (NDQ05) national completions agree
ndq = pd.read_csv(PROC / 'ndq05_completions_quarterly_la.csv')
ndq['date'] = pd.to_datetime(ndq['date']); ndq['year'] = ndq['date'].dt.year
qnat = ndq[ndq['area'] == 'Ireland'].groupby('year')['value'].sum()
cpm = pd.read_csv(PROC / 'ndm01_completions_monthly.csv')
cpm['date'] = pd.to_datetime(cpm['date']); cpm['year'] = cpm['date'].dt.year
mnat = cpm.groupby('year')['total'].sum()
chk = pd.concat([mnat.rename('monthly_ndm01'), qnat.rename('quarterly_ndq05')], axis=1).astype(int)
chk['abs_diff'] = (chk['monthly_ndm01'] - chk['quarterly_ndq05']).abs()
print('(a) completions cross-check (max abs diff = %d)' % chk['abs_diff'].max())
print(chk.loc[2011:2026].to_string())

# (b) RPPI base year 2015 = 100
rp = pd.read_csv(DER / 'rppi_monthly_index.csv')
rp['date'] = pd.to_datetime(rp['date']); rp['year'] = rp['period'].str[:4].astype(int)
nat = rp[rp['series'] == 'National - all residential properties']
print()
print('(b) national RPPI 2015 annual mean = %.2f  (target 100)' % nat[nat['year'] == 2015]['index'].mean())

# (c) CPI rebased 2015 = 100
ci = pd.read_csv(PROC / 'cpm24_cpi_allitems_monthly.csv')
ci['date'] = pd.to_datetime(ci['date'])
print('(c) CPI 2015 mean = %.2f (target 100); CPI Jun-2026 = %.1f' % (
    ci[ci['date'].dt.year == 2015]['cpi_2015=100'].mean(),
    ci[ci['period'] == '2026-07']['cpi_2015=100'].iloc[0]))

# (d) HPA09 mean price sanity: 2025 mean vs median
tx = pd.read_csv(DER / 'transactions_annual_national.csv')
r25 = tx[tx['year'] == 2025].iloc[0]
print()
print('(d) 2025 mean price EUR %.0f | median EUR %.0f' % (r25['tx_mean_price'], r25['tx_median_price']))
print('    checks passed.')

(a) completions cross-check (max abs diff = 11)
      monthly_ndm01  quarterly_ndq05  abs_diff
year                                          
2011           6994             6994         0
2012           4911             4911         0
2013           4575             4575         0
2014           5518             5518         0
2015           7219             7219         0
2016           9726             9720         6
2017          14223            14218         5
2018          17782            17777         5
2019          21023            21012        11
2020          20483            20483         0
2021          20443            20443         0
2022          29609            29609         0
2023          32463            32463         0
2024          30124            30124         0
2025          36215            36215         0
2026          16679            16679         0

(b) national RPPI 2015 annual mean = 99.98  (target 100)
(c) CPI 2015 mean = 100.00 (target 100); CPI Jun

# Outputs produced by this pipeline

**data/processed/** (tidy tables, one row per cell)
- hpm09_rppi_tidy.csv — RPPI index and 1/3/12-month changes, 20 series, monthly
- ndm01_completions_monthly.csv — completions by type (total / single / scheme / apartment) + trailing 12 months
- ndq05_completions_quarterly_la.csv — quarterly completions for Ireland and 31 local authorities
- hpa09_transactions_annual.csv — full HPA09 transaction cube (volume / value / mean / median)
- cpm24_cpi_allitems_monthly.csv — CPI All Items, rebased to 2015 = 100

**data/derived/** (analysis-ready, *new datasets generated for this project*)
- rppi_monthly_index.csv — cleaned price dataset (index + changes, parsed area/type)
- new_dwelling_completions_monthly.csv — cleaned monthly supply by type
- transactions_annual_national.csv — headline national annual transaction indicators
- transactions_annual_summary.csv — volume/value/mean/median by dwelling status x type
- housing_market_panel_quarterly.csv — **integrated quarterly market panel** (prices, CPI, real prices,
  completions incl. trailing-12m, transactions), 2011Q1-2026Q2 — the new dataset generated from the originals